# KnowDelay - Flight Delay Prediction — Phase 1 Report


## Project Members

In [0]:
import base64
from IPython.display import display, HTML

members = [
    {"name": "Priyanka Banerjee", "email": "pinkaban@berkeley.edu", "file": "/dbfs/FileStore/team_3_1_photos/PriyankaPhoto.jpeg"},
    {"name": "Eyup Agtepe", "email": "eyupagtepe@berkeley.edu", "file": "/dbfs/FileStore/team_3_1_photos/EyupPhoto.jpg"},
    {"name": "Deepak Srivastava", "email": "deepak_srivastava@berkeley.edu", "file": "/dbfs/FileStore/team_3_1_photos/Deepak.png"},
    {"name": "Carol Sanchez Garibay", "email": "carolsg@berkeley.edu", "file": "/dbfs/FileStore/team_3_1_photos/CarolPhoto.jpeg"},
    {"name": "Sunil Thakur", "email": "sunil.thakur@berkeley.edu", "file": "/dbfs/FileStore/team_3_1_photos/SunilPhoto.jpeg"}
]

def encode_img(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

html_output = """
<div style="display:flex; flex-wrap:wrap; justify-content:center; gap:28px; margin-top:20px; font-family:Arial;">
"""

for m in members:

    img = encode_img(m["file"])
    ext = m["file"].split(".")[-1]
    mime = "image/png" if ext == "png" else "image/jpeg"

    html_output += f"""
    <div style="text-align:center; width:190px;">
        <div style="width:140px;height:140px;margin:0 auto 12px;border-radius:50%;overflow:hidden;border:4px solid #003262;">
            <img src="data:{mime};base64,{img}" style="width:100%;height:100%;object-fit:cover;">
        </div>
        <div style="font-size:15px;font-weight:700;">{m['name']}</div>
        <div style="font-size:12px;color:#666;">{m['email']}</div>
    </div>
    """

html_output += "</div>"

display(HTML(html_output))


### Phase Leader Plan



| Week / Dates | Task  | Primary  | Secondary Support |
|------|-------------|-------------|---------|
| Week 0: 3/08-3/15  | Phase I | Eyup Agtepe | Priyanka Banerjee |
| Week 1: 3/15-3/22  | Phase II | Deepak Srivastava | Priyanka Banerjee |
| Week 2: 3/22-3/29  | Phase II | Deepak Srivastava | Priyanka Banerjee |
| Week 3: 3/29-4/05  | Homework 5  | Carol Sanchez Garibay | Eyup Agtepe |
| Week 4: 4/05-4/12  | Phase III | Sunil Thakur | Carol Sanchez Garibay |
| Week 5: 4/12-4/19  | Phase III  | Sunil Thakur | Carol Sanchez Garibay |

# Project Plan

The objective of this project is to develop a machine learning model capable of predicting flight delays using historical airline and weather data. The **primary stakeholders** are airlines, which can use predictions to provide timely passenger notifications, and airport operations such as air traffic control and ground support teams that must plan aircraft routing and takeoff logistics. The analysis uses the joined OTPW dataset, combining airline operational records from the U.S. Department of Transportation (DOT) with meteorological observations from the National Oceanic and Atmospheric Administration. Initial exploratory analysis of the three-month sample dataset reveals a mixture of categorical variables and numerical variables. Several weather fields contain missing or inconsistent values and require cleaning and normalization. Preliminary summary statistics also indicate large variance in delay times and potential relationships between delays and factors such as weather severity, departure time, and airport congestion. The **target variable** will be whether a flight is delayed by more than 15 minutes. Model performance will primarily be evaluated using the F₂ score, which places greater emphasis on recall in order to minimize missed delay predictions that could impact passenger communication and airline operations.

We first frame the problem as a **classification task**, using Logistic Regression as a baseline model to predict whether a delay will occur. Feature engineering will include handling missing values, encoding categorical variables, and scaling numerical inputs based on training statistics to prevent data leakage. Secondly, we will fit a more sophisticated model: Multi-layer Perceptron Classifier (MLP) that trains a feedforward artificial neural network and that will be evaluated with the same metrics as the baseline.

We will later extend the analysis to a **regression framework** to estimate the magnitude of flight delays using Linear Regression as a secondary baseline and model performance will primarly be evaluated using the Mean Squared Error (MSE) but we will have secondary metrics such as Root Mean Squared Error (RMSE), Mean Absolute Error (MAE) and Coefficient of Determination (R-Squared).

Time-aware training, validation, and testing splits will be used to prevent temporal leakage and ensure realistic model evaluation.


## 1. Data Description (EDA)

We analyzed four datasets to predict flight delays: the **OTPW** dataset (combined flights and weather data), the **Flights** dataset, the **Weather** dataset, and the **Airports** dataset. All exploratory analysis was conducted using the three-month sample datasets provided for the project.

Initial analysis revealed several important **data quality** characteristics. The Flights dataset contains duplicated rows that will be removed prior to modeling to prevent bias and inflated sample sizes. Additionally, many datasets contain columns with extremely high missingness. In the Flights and Weather datasets, several columns contain more than 99% missing values, mostly associated with events that occur rarely, and will therefore be removed to reduce dimensionality.

The exploratory analysis also identified variables that introduce **data leakage**, including columns that contain realized delay outcomes or operational information not available prior to departure. These variables will be excluded from the modeling dataset to ensure realistic prediction scenarios. Outlier detection on key numeric variables (such as delays and weather measurements) identified extreme observations, which will be evaluated and potentially capped or filtered depending on their operational interpretation.

Across the datasets, several **meaningful patterns** emerged from the exploratory visualizations. Flight delays exhibit a strong right-skewed distribution, where most flights depart near schedule but a smaller number experience very large delays. Delay probability varies significantly by airline carrier, departure time, and origin airport, with delay rates increasing throughout the day as operational disruptions propagate across the network. Weather conditions such as reduced visibility, precipitation, and high wind speeds also show potential relationships with delays. These findings motivated the inclusion of engineered features capturing temporal patterns, airport characteristics, airline behavior, and weather conditions.

To support efficient large-scale processing and iterative experimentation, the cleaned and transformed **datasets will be checkpointed** by writing intermediate tables to Parquet format. This approach improves I/O performance, reduces recomputation costs during repeated analysis, and ensures consistent dataset versions for model training and validation.

### 1.1. OTPW dataset

The On-Time Performance with Weather (OTPW) dataset combines U.S. domestic flight operations data from the U.S. Department of Transportation (DOT) with meteorological observations from the National Oceanic and Atmospheric Administration (NOAA). For this project, we use a three-month subset of 2015 (January–March) containing approximately **1.4 million observations** and **216 variables**.

This following diagram captures the full workflow for the OTPW analysis, starting from raw data to baseline modeling, evaluation, insights, and planning the next phase for extended analysis and modeling.

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/OTPW_EDA/eda_otpw_steps.png">

Each record represents a scheduled flight and includes information about airline operations, airport locations, route characteristics, scheduling information, and weather conditions observed near the origin airport prior to departure. The **target variable** is *DEP_DEL15*, a binary indicator equal to 1 if a flight departs at least 15 minutes late. The distribution of the target variable indicates that approximately **20% of flights experience departure delays of 15 minutes or more**, while the majority of flights depart on time. 

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/OTPW_EDA/sthakur_phase1_eda_overview_target.png">

A small portion of records contains missing delay labels and was excluded from modeling. The resulting **class distribution exhibits moderate imbalance**, which is typical for airline delay datasets and motivates the use of evaluation metrics such as F1 score and recall rather than accuracy alone.

#### 1.1.1 Data Quality and Validation

Initial exploratory analysis focused on validating the integrity and structure of the OTPW dataset. Sanity checks were performed to identify invalid values, inconsistent formats, and potential outliers across scheduling, weather, and operational variables. Data type validation confirmed that temporal variables (e.g., month, day, scheduled departure time), numerical weather measurements, and categorical identifiers (e.g., carrier, airport codes) were correctly represented after cleaning and transformation. **Missing value analysis** revealed that most features had negligible missingness, with the exception of HourlyPrecipitation (~11%), which was subsequently imputed using a median-based strategy to preserve distributional characteristics. Outlier inspection of weather variables and flight distance confirmed that values generally fall within realistic operational ranges.

Missing value analysis shows that a large subset of features contains extremely high missingness, with several columns having over 99% missing values and some containing no valid observations. These variables largely correspond to rarely reported weather measurements and station metadata fields. After grouping features by missing-value percentage ranges, we observe that most informative variables fall within the 0-10% missing range, while columns with over 75% missing values provide limited analytical value. 

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/OTPW_EDA/sthakur_phase1_eda_missing_value_analysis.png">

Consequently, features with extremely high missingness were removed from further analysis, while columns with moderate missingness were addressed through targeted imputation strategies.

As part of data validation, we conducted an outlier analysis across all numerical variables using the interquartile range (IQR) method. This analysis helps identify potential data quality issues, sensor anomalies, or extreme observations that may influence model training.

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/OTPW_EDA/sthakur_phase1_edh_outliers_analysis.png">

The results indicate that a small subset of variables, primarily weather-related features such as wind speed, precipitation, and visibility—exhibit higher outlier percentages. These values are largely attributable to genuine extreme weather conditions rather than data errors, and therefore were retained as they provide meaningful predictive signal for flight delays. 

#### 1.1.2 Temporal Patterns

We next analyzed the distribution and temporal dynamics of the target variable, **DEP_DEL15**, which indicates whether a flight departed at least 15 minutes late. The dataset exhibits a moderate class imbalance, with delayed flights representing a minority of observations. Temporal analysis revealed clear patterns across the daily and monthly flight schedule. Delay rates increase during peak departure hours, particularly during late afternoon and evening periods when operational congestion is highest. Monthly flight volumes remain relatively stable across January through March, though delay rates fluctuate slightly, potentially reflecting seasonal weather variability and operational demand.

Temporal analysis reveals strong patterns in flight delay behavior. Delay probability increases steadily throughout the day, rising from approximately 7% in early morning departures to nearly 28% in late evening hours, reflecting the cumulative impact of operational disruptions and congestion. 

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/OTPW_EDA/sthakur_phase1_eda_temmporal_analysis_delay_rate.png">

Weekly variation is moderate, with slightly higher delay rates observed on certain weekdays, potentially linked to higher traffic volumes. Monthly differences between January, February, and March remain relatively small, although February exhibits a slightly higher delay rate, which may reflect seasonal weather conditions. These observations motivated the inclusion of engineered temporal features such as departure hour and cyclical time encodings in the predictive model.

Flight volume analysis reveals clear temporal patterns in airline operations. Departure activity increases sharply during early morning hours, peaks during mid-day and afternoon periods, and gradually declines during late evening hours. 

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/OTPW_EDA/sthakur_phase1_eda_temporal_analysis_flight_volume.png">

Weekly patterns show relatively consistent traffic levels across days of the week, while monthly flight volumes remain broadly similar across the three-month observation period. These traffic patterns provide important context for understanding delay behavior, as periods of higher operational demand are more susceptible to congestion-related disruptions.

#### 1.1.3 Operational Factors: Airlines, Airports, and Routes

Operational characteristics of airlines, airports, and routes were examined to identify structural contributors to delays. Airline-level analysis indicates noticeable variation in delay rates across carriers, reflecting differences in operational efficiency, fleet utilization, and route networks. Airport-level analysis highlights major hub airports with the highest traffic volumes, where congestion effects may influence delay probability. Route-level exploration further revealed that certain origin–destination pairs exhibit systematically higher delay rates, often corresponding to heavily trafficked corridors or airports with limited operational slack. 


<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/OTPW_EDA/sthakur_phase1_eda_operationa_analysis_1.png">


<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/OTPW_EDA/sthakur_phase1_eda_operational_analysis_2.png">

These findings motivated the inclusion of engineered features such as route-level and airport-level historical delay rates to capture persistent operational patterns.

#### 1.1.4 Weather Conditions and Delay Relationships

Weather variables were explored to understand their relationship with departure delays. Key meteorological indicators—including precipitation, wind speed, visibility, and temperature show meaningful variation across flights and potential associations with delays. Preliminary analysis suggests that adverse weather conditions, particularly low visibility, precipitation, and strong winds, correlate with elevated delay rates. 

Boxplot analysis of key weather variables shows measurable distributional differences between delayed and on-time flights. In particular, higher wind speeds, reduced visibility, and increased precipitation appear more frequently in delayed flights, suggesting that adverse weather conditions may contribute to operational disruptions. 


<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/OTPW_EDA/sthakur_phase1_eda_weather_variables.png">

These observations motivated the inclusion of weather-derived features and interaction variables in the predictive modeling pipeline. To better capture weather-related risk factors, additional engineered features such as binary severe-weather indicators and interaction variables combining wind speed and visibility were introduced. These transformations allow the model to represent nonlinear weather impacts more effectively.

####1.1.5 Airline Carrier and Delay Rate/Flights Count Relatinship

Across carriers, on‑time performance is far from uniform. Some airlines show noticeably higher average arrival delays, while others manage to stay much closer to schedule. However, once we factor in the number of flights each carrier operates, it becomes clear that a few high‑volume airlines have an outsized impact on passengers’ overall experience: even if their average delay is only moderate, the sheer scale of their operations means they account for a large share of all delayed arrivals.

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/OTPW_EDA/carrier%20vs%20avg%20delay.jpg" width="1020"/>

#### 1.1.6 Feature Diagnostics for Modeling

Following initial data validation and exploratory analysis, we constructed a candidate feature set for modeling after applying several filtering and cleaning steps. Columns with extremely high missingness and excessively high cardinality were removed to avoid noise, sparsity, and unnecessary model complexity. Additionally, potential data leakage features were manually identified and excluded by reviewing variables whose values are determined after the flight departure event. Basic data cleaning was also performed to standardize column types and handle missing values. For numerical variables, missing values were imputed with 0, while for categorical variables missing values were replaced with Null to allow downstream encoding. Based on this filtered dataset and insights from the limited EDA such as temporal delay patterns, airport and airline variability, route characteristics, and weather-related impacts, we selected a focused set of features for feature engineering and baseline modeling.


| Feature Category                  | Feature Name                | Description                                      |
| --------------------------------- | --------------------------- | ------------------------------------------------ |
| **Temporal Features**             | `QUARTER`                   | Quarter of the year                              |
|                                   | `MONTH`                     | Month of flight                                  |
|                                   | `DAY_OF_MONTH`              | Day of month                                     |
|                                   | `DAY_OF_WEEK`               | Day of week                                      |
|                                   | `CRS_DEP_TIME`              | Scheduled departure time                         |
|                                   | `CRS_ARR_TIME`              | Scheduled arrival time                           |
| **Categorical Features**          | `OP_UNIQUE_CARRIER`         | Operating airline carrier                        |
|                                   | `ORIGIN`                    | Origin airport                                   |
|                                   | `DEST`                      | Destination airport                              |
|                                   | `CANCELLED`                 | Flight cancellation indicator                    |
|                                   | `DIVERTED`                  | Flight diversion indicator                       |
|                                   | `route`                     | Origin–destination route identifier              |
| **Route / Geographic Features**   | `origin_station_lat`        | Latitude of origin weather station               |
|                                   | `origin_airport_lat`        | Latitude of origin airport                       |
|                                   | `dest_station_lat`          | Latitude of destination weather station          |
|                                   | `dest_airport_lat`          | Latitude of destination airport                  |
| **Weather Features**              | `HourlyAltimeterSetting`    | Atmospheric pressure measurement                 |
|                                   | `HourlyDewPointTemperature` | Dew point temperature                            |
|                                   | `HourlyDryBulbTemperature`  | Air temperature                                  |
|                                   | `HourlyPrecipitation`       | Precipitation amount                             |
|                                   | `HourlyPressureTendency`    | Pressure change over time                        |
|                                   | `HourlyRelativeHumidity`    | Relative humidity                                |
|                                   | `HourlySeaLevelPressure`    | Pressure at sea level                            |
|                                   | `HourlyStationPressure`     | Station pressure                                 |
|                                   | `HourlyVisibility`          | Visibility conditions                            |
|                                   | `HourlyWetBulbTemperature`  | Wet bulb temperature                             |
|                                   | `HourlyWindDirection`       | Wind direction                                   |
|                                   | `HourlyWindGustSpeed`       | Wind gust speed                                  |
|                                   | `HourlyWindSpeed`           | Wind speed                                       |
| **Derived / Engineered Features** | `dep_hour`                  | Hour extracted from scheduled departure time     |
|                                   | `wind_bin`                  | Categorized wind speed                           |
|                                   | `distance_bin`              | Binned flight distance                           |
| **Target Variable**               | `DEP_DEL15`                 | Binary indicator of departure delay ≥ 15 minutes |


Correlation analysis among numerical features revealed limited multicollinearity, with most weather variables demonstrating weak to moderate relationships with the target variable.

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/OTPW_EDA/sthakur_phase1_eda_correlation_analysis.png">

Cardinality analysis highlighted that certain categorical variables such as airports and routes that contain a large number of unique values, motivating the use of encoding strategies and aggregated historical delay features.


Additional predictive signal tests confirmed that engineered features such as departure hour, distance bins, route delay rate, and airport delay rate provide meaningful discriminatory power between delayed and on-time flights. 

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/OTPW_EDA/sthakur_phase1_eda_engineered_features.png">

#### 1.1.7 Validating the EDA with Baseline Model

To validate whether the insights obtained from exploratory data analysis (EDA) translate into predictive signal, we trained a baseline classification model using Logistic Regression. The objective of this model is not to achieve the best performance but to verify that the selected features—derived from temporal patterns, operational characteristics, route attributes, and weather conditions—contain meaningful information for predicting flight departure delays. Because the dataset spans only three months of data for a given year, we adopted a time-aware train–test split to avoid temporal leakage and better simulate a real-world deployment scenario. Specifically, flights from the first two months were used for training, while flights from the third month were held out for testing. This approach ensures that the model learns from historical data and is evaluated on future observations, reflecting how delay prediction systems would operate in practice. After applying the feature engineering pipeline and training the logistic regression model, the baseline model achieved a test accuracy of 68.83% and an AUC of 68.85%. 


<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/OTPW_EDA/sthakur_phase1_edh_baseline_model.png">

These results suggest that the selected feature set captures a meaningful portion of the variability in flight delays, providing a reasonable starting point for further model refinement, feature engineering, and experimentation with more advanced algorithms in subsequent phases of the project.

### 1.2. Flights Dataset

#### 1.2.1 Data Quality and Validation

The exploratory data analysis was conducted on the three-month sample flight dataset, which contains **2,806,942 observations** and **109 variables** describing flight operations, scheduling, airport information, and delay outcomes. 

Initial data quality checks revealed **1,403,471 duplicated rows**, which represent exactly half of the dataset and will be removed prior to model training to avoid bias and inflated sample size. Additionally, 47 variables contain more than 99% missing values, mostly related to diversion-related fields that occur very rarely; these columns provide little predictive value and will therefore be dropped to reduce dimensionality and improve computational efficiency. 


#### 1.2.2 Leakage analysis

During feature inspection, **10 variables contain future information** (such as realized delays and post-flight delay breakdowns). These variables leak information that would not be available at prediction time and must be excluded from the modeling dataset. Finally, an outlier detection rule applied to delay-related numeric variables identified **315,610 extreme observations**, which will be carefully evaluated.

Overall, the **target variable** (DEP_DEL15) have the same behavior mentioned in the OTPW dataset.The distribution of departure delays is heavily right-skewed, with most flights departing near schedule and a long tail of extreme delays. 

#### 1.2.3 Target variable across variables

Temporal patterns emerge: delay probability increases throughout the day, peaking in the evening hours as delays accumulate across the network, while early morning departures experience the lowest delay rates.

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/FLIGHTS_EDA/flights_delayed_distribution_by_departure_time.png">

Delay rates vary significantly across airlines, with some carriers exhibiting delay rates above 30%, while others remain below 10%, suggesting operational differences among carriers. 

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/FLIGHTS_EDA/flights_delayed_distribution_by_carrier.png">

These Exploratory Data Analysis findings suggest that **carrier, departure time, day-of-week, and origin airport characteristics are likely to be informative predictors** in the modeling phase.

### 1.3. Weather Dataset
Similarly to the OTPW and flight datasets, the 3-month weather dataset covers the period from January to March. It contains 30,528,602 records and 124 columns. The dataset includes a variety of features describing weather conditions, such as hourly wind speed, hourly visibility, and hourly wind direction. Since this dataset also has a large number of columns, we decided to first take a 1% sample of the data and then analyze it.

##### 1.3.1 Missing Data Analysis
As mentioned above, the dataset contains nearly 31 million rows and 124 columns. Performing a quick EDA on a dataset of this size would be challenging. After sampling the data, we checked the missing values column by column. We found that 105 out of 124 columns had more than half of their values missing. Since imputing these columns would be difficult and potentially unreliable, we decided to drop them.

Finally, we dropped the year column for this EDA because the same information is already available in the DATE column. After removing the mostly null-valued columns and the year column, we were left with 18 columns:

| Column                   | Description                                                                                      |
|--------------------------|--------------------------------------------------------------------------------------------------|
| STATION                  | Unique weather station identifier (e.g. WBAN or ICAO code)                                       |
| DATE                     | Timestamp of the weather observation                                                             |
| LATITUDE                 | Station's north-south position in decimal degrees                                                |
| LONGITUDE                | Station's east-west position in decimal degrees                                                  |
| ELEVATION                | Station's vertical distance above sea level in meters                                            |
| NAME                     | Station name with city and country/state                                                         |
| REPORT_TYPE              | Type of weather observation (e.g. FM-15 for METAR, FM-16 for SPECI)                             |
| SOURCE                   | Numeric code indicating the data source/provider                                                 |
| HourlyAltimeterSetting   | Atmospheric pressure adjusted to sea level, in inches of mercury                                 |
| HourlyDewPointTemperature| Temperature at which air becomes saturated, in °F                                                |
| HourlyDryBulbTemperature | Ambient air temperature measured by a standard thermometer, in °F                                |
| HourlyRelativeHumidity   | Percentage of moisture in the air relative to saturation                                         |
| HourlySkyConditions      | Cloud cover description (e.g. FEW, SCT, BKN, OVC with altitude)                                 |
| HourlyStationPressure    | Atmospheric pressure at the station's elevation, in inches of mercury                            |
| HourlyVisibility         | Horizontal distance at which objects can be seen, in statute miles                               |
| HourlyWindDirection      | Compass direction the wind is blowing from, in degrees 0-360                                     |
| HourlyWindSpeed          | Speed of sustained wind, in miles per hour                                                       |
| REM                      | Raw alphanumeric remarks text transmitted by the weather station                                 |

**Note:** During the duplicate row analysis, we did not identify any duplicate rows in the 1% sample of the dataset.

##### 1.3.2 Outlier Detection
Similar to the flight dataset EDA, we used the IQR method to detect outliers in the numeric columns, except for LONGITUDE, LATITUDE, ELEVATION, and HourlyStationPressure. We excluded LONGITUDE and LATITUDE because they are location identifiers rather than variables suitable for outlier detection. Similarly, ELEVATION represents the station’s height above sea level and reflects the geographic characteristics of the station’s location. HourlyStationPressure was also excluded because it is highly correlated with elevation, as high-altitude areas generally have lower atmospheric pressure than locations at sea level. For the remaining numeric columns, values below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR were flagged as outliers. Rows with NaN values were preserved and were not treated as outliers.

| Column                | PCT Outlier           | Count        | Potential Root Cause of Outliers           |
|--------------------------|---------------|----------------------|-----------------------|
| hourly_visibility      | 7.43%| 14,762  | 14,762	Low-visibility events (fog, storms) generally push values outside of typical range    |
| hourly_wind_speed	      | 3.53%    | 9,349   | Extreme gusts from severe weather events         |
| hourly_altimeter_setting    | 3.34%| 5,483| Pressure anomalies during major weather systems |
| hourly_dew_point_temp            | 2.37%         | 5,958         |Extreme humidity or very dry conditions    |
| hourly_dry_bulb_temp            | 2.19%         | 6,536	         |Heat waves or cold snaps    |
|hourly_relative_humidity |	1.49%	| 3,732 |	Mostly within expected range (0–100%)|
|hourly_wind_direction	|0.00%	|0	|Circular scale (0–360°) — IQR range spans nearly entire domain, so no outliers detected|


##### 1.3.3 Cardinality Analysis
The cardinality ratio, defined as the number of unique values divided by the total number of rows, revealed that the REM column has very high cardinality, with a score of 0.8575. This means that nearly every value in this column is unique text. High cardinality may indicate that the column will be difficult to encode using standard categorical encoding methods. As a result, we may eventually need to drop the REM column or instead apply NLP techniques or text parsing and feature extraction methods.

HourlySkyConditions has a moderate cardinality ratio of 0.09. The STATION and NAME columns have relatively low cardinality ratios of 0.03, while REPORT_TYPE and SOURCE have extremely low ratios of 0.0004. These lower-cardinality columns may be suitable candidates for one-hot encoding in future machine learning model development.


##### 1.3.4 Value Distribution
In this section, we detail the distribution characteristics, skewness, and outlier presence for key weather and station variables based on histogram and boxplot analyses.

##### 1.3.4.1 Station Characteristics

**Elevation**
- **Distribution:** J-shaped, heavily right-skewed.
- **Outliers:** Values exceeding 1,000 feet are sparse and appear as outliers.
- **Insight:** Potential to engineer a categorical or binary feature for extreme high-elevation stations.

**Temperature & Moisture Variables**
- **Hourly Dew Point Temperature**
  - **Distribution:** Unimodal, slightly left-skewed; central spread between 20 and 45.
  - **Outliers:** Extreme negative values pull the distribution left; boxplot shows multiple low-end outliers.
- **Hourly Dry Bulb Temperature**
  - **Distribution:** Unimodal, slight left skew; centered between 25 and 60.
  - **Outliers:** Present on both low and high extremes.
- **Hourly Relative Humidity**
  - **Distribution:** Left-skewed; most values between 60 and 90.
  - **Outliers:** Several extreme low-end outliers.

**Atmospheric Pressure Variables**
- **Hourly Altimeter Setting**
  - **Distribution:** Unimodal; median at 30.2 hPa.
  - **Outliers:** Primarily on the left side, indicating extreme low values.
- **Hourly Station Pressure**
  - **Distribution:** Left-skewed; tight central spread near upper range (28–30).
  - **Outliers:** Numerous low-value outliers.

**Wind & Visibility Variables**
- **Hourly Wind Speed**
  - **Distribution:** Strong right-skew; most values are low, with a long tail toward higher speeds.
  - **Outliers:** Many high-end outliers, indicating extreme wind events.
- **Hourly Wind Direction**
  - **Distribution:** Widely dispersed; concentration near 0°.
  - **Outliers:** No major extreme outlier behavior.
- **Hourly Visibility**
  - **Distribution:** Strong right-skew; clustered at lower values, long tail to higher visibility.
  - **Outliers:** Numerous high-end outliers, suggesting occasional extreme visibility events.

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/main/Phase%20I%20Report%20Images/Visualizations/Weather_EDA_Images/weather_column_distribution_50.png">

##### 1.3.5 Correlation Analysis
In our correlation analysis of the numeric values in the weather dataset, we found that most variable pairs were only weakly correlated, which suggests there is not a lot of strong linear overlap among the predictors. Two pairs stand out clearly. **hourly_dew_point_temp** and **hourly_dry_bulb_temp** have a very strong positive correlation **(0.91)**, meaning they tend to increase together and may carry very similar information. **Elevation** and **hourly_station_pressure** have a very strong negative correlation **(-0.98)**, which shows that station pressure tends to decrease as elevation increases. These two pairs are the main ones to watch because they could create multicollinearity issues in models like linear regression.

Aside from those relationships, the rest of the variables are only weakly related. For example, **hourly_relative_humidity** and **hourly_visibility** show a small negative correlation **(-0.26)**, while **hourly_wind_direction** and **hourly_wind_speed** have a weak positive correlation **(0.33)**. Overall, the correlation matrix suggests that most features contribute fairly distinct information, with only a few pairs showing substantial overlap.

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/main/Phase%20I%20Report%20Images/Visualizations/Weather_EDA_Images/weather_cor_heatmap_50.png">


### 1.4. Airport Datasets

The airport dataset under analysis consists of 5,004,169 rows and 12 columns, including 5 numeric columns (lat, lon, neighbor_lat, neighbor_lon, distance_to_neighbor) and 7 categorical columns (usaf, wban, station_id, neighbor_id, neighbor_name, neighbor_state, neighbor_call). The dataset is remarkably clean, with zero missing values across all columns, and the numeric fields exhibit consistent ranges with no obvious outliers.

##### 1.4.1 Summary Stats for Numeric Features

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/AIRPORT_DATA/sthakur_phase1_eda_airport_numeric.png">

##### 1.4.2 Summary Stats for Categorical Features

<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/AIRPORT_DATA/sthakur_phase1_eda_airport_categorical.png">

Exploring the airport dataset for graph feature engineering offers a promising avenue to capture network effects in flight delays. By considering each station as a node and its neighboring stations as connected nodes, we can model relationships such as distance to neighbor, spatial clustering, and regional traffic flows. These graph-based features could include centrality measures, neighbor delay propagation, and connected hub influences, which are likely to enhance predictive power when combined with the OTPW flight and weather data. This approach allows us to account for inter-airport dependencies, providing richer context for modeling delay patterns across the US domestic flight network.

## 2. Data Partitioning - Test/Train Strategy

1. Temporal Splitting Strategy: To ensure our model generalizes to future flights, we will be avoiding a random split in favor of a Chronological Partitioning strategy. By separating data by year and quarter, we simulate a real-world scenario where a model is trained on historical records to predict "unseen" future schedules. <br>

  - Training & Validation Window (2015–2018): We plan to utilize the full 16 quarters of data from January 2015 through December 2018 for our model development phase.

  - Training Set: Approximately the first 80-85% of this period (2015 through roughly mid-2018) will be used to learn the foundational relationships between weather patterns and carrier logistics.

  - Validation Set: We will use the final quarters of 2018 serve as our "developmental future." This allows us to tune hyperparameters and perform early stopping without touching our final evaluation data.

  - Final Test Set (2019): All four quarters of 2019 will be strictly reserved as the Held-out Test Set. By using an entire distinct year for testing, we can evaluate if the model is robust enough to handle the seasonal shifts and unique operational changes of a completely new calendar year.

2. Time Series Cross-Validation (Expanding Window)Since flight delays are essentially a time-series problem, standard K-Fold Cross-Validation is unsuitable because it shuffles data across time. Instead, we will use an Expanding Window approach within our 2015–2018 training block.In this configuration, we will iterate through our quarters:

  - Fold 1: Train on 2015 Q1 -> Validate on 2015 Q2.

  - Fold 2: Train on 2015 Q1–Q2 -> Validate on 2015 Q3.

  - Fold 3: Train on 2015 Q1–Q3 -> Validate on 2015 Q4.<br><br>This "step-forward" logic ensures that at no point does the model look at "future" quarters to predict "past" delays, preserving the chronological integrity of our features.


### 2.1 Machine Learning Algorithms and Metrics

For this flight delay prediction project, the delay that we are going to predict will be defined as any delay over 15 minutes. We will initially approach the problem as a binary classification task to predict whether a flight will be delayed (yes/no), using Logistic Regression as our baseline model. This will allow us to establish a foundational understanding of the predictive power of our features. Subsequently, we will transition to a regression task to predict the magnitude of the delay (in minutes), using Linear Regression as the baseline, followed by Ridge Regression to address potential multicollinearity in our features (e.g., correlated weather or flight variables). This two-phase approach enables us to first assess binary delay occurrence and then quantify delay severity.


#### 2.1.1 Logistic Regression (Classification Phase):

We will use Logistic Regression, implemented via the LogisticRegression class from MLLib. This implementation supports regularization (L1/L2) to prevent overfitting.

**Loss Function:** The loss function is binary cross-entropy (also known as log loss), which measures the difference between predicted probabilities and actual binary outcomes. The formula is:
$$
L = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]
$$

where n is the number of samples, \\(y_i) \text{ is the true label (0 or 1), and } \hat{y}_i\\)  is the predicted probability.


**Metrics:**

* **F2-Score**: A variant of the F1-score that gives more weight to recall than precision (with β = 2), helping to reduce false positives by prioritizing the correct identification of delays to ensure passengers are in the airport. <br> 
$$ F2 = \frac{5 * Precision * Recall}{4 * Precision + Recall} $$.
* **ROC Curve / AUC**: The Receiver Operating Characteristic (ROC) curve plots Recall (TPR) vs False Positive Rate (FPR) across all classification thresholds. It helps us understand how changing the decision threshold impacts the trade-off between catching delays and raising false alarms. \
It allows selecting a threshold that matches flight delays operational goals (e.g., prefer higher recall even if false positives rise, or balance both).
AUC (Area Under the Curve): Summarizes the model's ability to distinguish delayed vs on-time flights independent of any specific threshold; higher AUC indicates better overall separability.

$$
\text{TPR (Recall)} = \frac{TP}{TP + FN}
$$
$$
\text{FPR} = \frac{FP}{FP + TN}
$$
* **Recall** (Sensitivity): The proportion of actual positives correctly identified. We are about this as we want to minimize false positives (false reports of delays as that could potentially cause monetary losses for the companies) <br>
$$ Recall = \frac{TP}{TP + FN} $$.
* **Accuracy**: The proportion of correct predictions. <br>
$$ Accuracy = \frac{TP + TN}{TP + TN + FP + FN} $$ 
where TP = true positives, TN = true negatives, FP = false positives, FN = false negatives. It helps provide more of a flat value for our overall correct predictions of delays.
* **Precision**: The proportion of positive predictions that are correct. It helps us determine a direct proportion of correct delay predictions <br>
$$ Precision = \frac{TP}{TP + FP} $$



#### 2.1.2 Linear Regression (Regression Phase - Baseline):

We will use Linear Regression, implemented via the LinearRegression class from MLLib. This uses ordinary least squares (OLS) for fitting the model and supports feature scaling for better convergence.

**Loss Function:** The loss function is Mean Squared Error (MSE), which quantifies the average squared difference between predicted and actual values. 
$$ L = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 $$
where \\( n\\) is the number of samples, \\( y_i\\) is the actual delay (in minutes), and \\( \hat{y}_i\\) is the predicted delay.

**Metrics:** 

* Mean Squared Error (MSE): Average squared prediction error. Squaring heavily penalizes large errors, which is important because large delay prediction errors can lead to significant operational costs (e.g., staffing, gate reassignments, and passenger accommodations). <br> 
$$ MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 $$
* Root Mean Squared Error (RMSE): Square root of MSE, in the same units as the target. Because RMSE is in minutes, it’s easier for stakeholders to interpret and understand how far off predictions are in real-world terms (e.g., “we’re typically off by X minutes”). 
$$ RMSE = \sqrt{MSE} $$.
* Mean Absolute Error (MAE): Average absolute prediction error. MAE provides a clear measure of average prediction error without exaggerating outliers, which helps estimate typical operational impacts on scheduling and customer communication.
$$ MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i| $$
* R-Squared (Coefficient of Determination): Proportion of variance explained by the model. R² indicates how much of the delay variability the model captures; higher values mean the model is better at explaining why delays happen, which supports better resource planning and decision-making.
$$ R^2 = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2} $$
where \\( \bar{y}\\) is the mean of actual values.

If we know that \\( a=2\\) and \\( b=3\\), we can calculate \\( c=\sqrt{a^2 + b^2}\\).

#### 2.1.3 Multilayer Perceptron Classifier (Classification Phase - Advanced):
To satisfy updated requirements and capture more complex relationships, we will use an MLP Classifier for the advanced classification phase. This provides a non-linear model that can learn interactions across features.

**Implementation:** Use MLLib's MultilayerPerceptronClassifier.
Training: Uses backpropagation to update weights across layers.
Loss Function: Uses logistic (cross-entropy) loss for optimization.
Optimizer: Uses L-BFGS for stable convergence when training.
Focus: Keep emphasis on F2-score to reduce false positives and ensure passengers get accurate delay notifications.

**Metrics:** (Same as Baseline Classification Metrics)

* **F2-Score**: A variant of the F1-score that gives more weight to recall than precision (with β = 2), helping to reduce false positives by prioritizing the correct identification of delays to ensure passengers are in the airport. <br> 
$$ F2 = \frac{5 * Precision * Recall}{4 * Precision + Recall} $$
* **ROC Curve / AUC**: The Receiver Operating Characteristic (ROC) curve plots Recall (TPR) vs False Positive Rate (FPR) across all classification thresholds. It helps us understand how changing the decision threshold impacts the trade-off between catching delays and raising false alarms. \
It allows selecting a threshold that matches flight delays operational goals (e.g., prefer higher recall even if false positives rise, or balance both).
AUC (Area Under the Curve): Summarizes the model's ability to distinguish delayed vs on-time flights independent of any specific threshold; higher AUC indicates better overall separability.

$$
\text{TPR (Recall)} = \frac{TP}{TP + FN}
$$
$$
\text{FPR} = \frac{FP}{FP + TN}
$$
* **Recall** (Sensitivity): The proportion of actual positives correctly identified. We are about this as we want to minimize false positives (false reports of delays as that could potentially cause monetary losses for the companies) <br>
$$ Recall = \frac{TP}{TP + FN} $$
* **Accuracy**: The proportion of correct predictions. <br>
$$ Accuracy = \frac{TP + TN}{TP + TN + FP + FN} $$ 
where TP = true positives, TN = true negatives, FP = false positives, FN = false negatives. It helps provide more of a flat value for our overall correct predictions of delays.
* **Precision**: The proportion of positive predictions that are correct. It helps us determine a direct proportion of correct delay predictions <br>
$$ Precision = \frac{TP}{TP + FP} $$



## 3. Machine Learning Pipelines

[Please remove at end]
Description of the pipeline steps you plan to use
Create your ML pipeline: from data ingestion to model. You NEED to create a block diagram and attach it in the notebook.



<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/Implementation_Planning_Visuals/BlockDiagram.png">



### 3.1 Machine Learning Pipeline Structure
1. **Dataset: OTPW Dataset** \
Main dataset that we will be utilizing for the project. We will potentially switch over to a custom-joined dataset as necessary.

2. **ETL: Extract, Transform, Load** \
ETL processes the raw OTPW dataset. Extraction involves pulling data from sources; transformation includes cleaning (e.g., handling missing values, encoding categorical variables like airport codes), feature engineering (e.g., creating delay indicators or aggregating weather features), and normalization/scaling. Loading prepares the data for modeling by converting it into MLLib-compatible formats like VectorAssembler for features and label indexing. This step ensures the data is ready for machine learning algorithms.

3. **Train/Test Split** \
Divide the processed dataset into training and testing sets to evaluate model performance on unseen data. Typically, use an 80/20 or 70/30 split. In MLLib, this is done using randomSplit() on the DataFrame. The training set is used for model fitting, while the test set is held out for final evaluation to prevent overfitting.

4. **Time Series Cross Validation for Logistic Regression Training** \
Perform time series cross-validation (eg. 5 time-sequential folds) on the training set to tune hyperparameters and validate the Logistic Regression model. In MLLib, use CrossValidator with LogisticRegression as the estimator, along with a parameter grid for regularization (e.g., elasticNetParam). This step trains multiple models on different folds, averages performance metrics (e.g., accuracy, F2-score), and selects the best model to avoid overfitting and ensure generalization.

5. **Test Dataset Evaluation** \
Evaluate the trained Logistic Regression model on the held-out test dataset using metrics like accuracy, precision, recall, F2-score, and AUC-ROC. In MLLib, use MulticlassClassificationEvaluator or BinaryClassificationEvaluator to compute these. Analyze confusion matrices and ROC curves to assess how well the model predicts flight delays, ensuring it meets project goals like minimizing false negatives.

6. **Model Deployment/Iteration** \
Deploy the evaluated model into production (e.g., via MLLib's model saving/loading with model.save() and LogisticRegressionModel.load()). Monitor performance on new data and iterate by refining features, trying other algorithms (e.g., Random Forest), or retraining with updated datasets. This ensures the pipeline is continuous and adaptable.

### 3.2 Data Leakage Prevention Strategy

Standard random shuffling is the most common source of leakage in time-series data. By implementing a strictly chronological split, we will ensure the model only learns from historical data.

We will use 2015–2018 exclusively for training and validation.

We will hold out 2019 as a completely independent test set.
This prevents the model from "peeking" at future weather patterns or systemic airline delays that it wouldn't have access to in a live environment.

Since this is a joined dataset, we would be cautious of using variables which may be directly or indirectly associated and cause data leakage in our model.

- We plan to avoid using ARR_DELAY or AIR_TIME to predict DEP_DELAY. These are "future" values we wouldn't know it at the time of departure.
- Weather Timing: We will ensure we are using the weather columns associated with the four_hours_prior or two_hours_prior windows to build a realistic forecasting model. Using the weather at the exact time of departure is technically not forecasting.

We will compute all scaling parameters (Mean, StdDev, Min/Max) using only the Training/Validation window (2015–2018). These pre-computed statistics are then used to transform the 2019 test data.


# Implementation Plan


### Gantt Chart




<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/Implementation_Planning_Visuals/GanttChart.png">

### Member Assignment Plan
1. Each team member is planned to work on as follows:
    1. Priyanka will implement the OTPW data ETL, design the feature‑engineering and graph‑based feature pipelines, and build the inference and training/modeling workflows. In addition, she will create the initial report skeletons so the team’s data, methods, and results are organized consistently.
    2. Sunil will help implement the OTPW data ETL and feature‑engineering pipeline, then build baseline modeling pipelines for both the 3‑month and 1‑year datasets, including the definition and computation of evaluation metrics. He will also contribute to graph‑based feature construction and synthesize the main technical and business takeaways in the conclusion section of the report.
    3. Eyup will focus on model tuning and overall project framing. He will support the OTPW data ETL, then lead hyperparameter tuning via grid search to strengthen model performance. In parallel, he will assess the feasibility of the project, identify key risks and next steps, and author the project description section that clearly explains the problem, goals, and planned approach.
    4. Carol will focus on temporal features, scalability, and communication. She will design and implement time‑based features (e.g., schedules, seasonality signals) and help review the pipelines for efficiency and scalability. She will also lead the preparation of the team meta‑information and project abstract, and she will own the EDA results section, summarizing key patterns and visualizations from the data.
    5. Deepak will specialize in evaluation and validation strategy. He will implement appropriate train/test splits that respect the temporal nature of flight data and develop cross‑validation methods designed specifically for time‑series prediction. He will also contribute to the results section, documenting the model performance metrics, validation outcomes, and key findings from the evaluation process.
    3. As a team, we will need to integrate our pipelines,  results,  discussion, and  conclusions into one notebook.




### Credit Assignment Plan




<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/Implementation_Planning_Visuals/Phase1CreditAssignment.png">




<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/Implementation_Planning_Visuals/Phase2CreditAssignment.png">



<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/Implementation_Planning_Visuals/Phase3CreditAssignment.png">





<img src="https://raw.githubusercontent.com/Pinkaban/FlightDelayPrediction/refs/heads/main/Phase%20I%20Report%20Images/Visualizations/Implementation_Planning_Visuals/HW5Assignment.png">


# Planned Next Steps

## 1. Summary

In Phase 2, our objective will be to validate that our data contains sufficient predictive signal to forecast flight delays. To accomplish this, we will implement a first machine learning workflow using a classification model. The initial focus will be on building a baseline model that confirm if delay prediction is feasible.

The first step will be creating the **data preprocessing pipeline**. This includes cleaning missing records identified during the exploratory analysis, and standardizing feature formats. Categorical variables will be encoded, while numerical variables will be normalized using statistics computed *only* from the training dataset to avoid data leakage. The cleaned dataset will then be checkpointed to support efficient distributed processing.

Next, the dataset will be split into training, validation, and test sets using a *time-aware partitioning strategy* to prevent leakage across temporal boundaries.

Using this prepared dataset, we will **train a baseline** Logistic Regression classification model. This model will serve as a first benchmark to assess the predictive capability of the engineered features. Model performance will be evaluated using the F₂ score, which emphasizes recall and prioritizes correctly identifying delayed flights while minimizing missed delay predictions. Additional metrics such as precision, recall, and confusion matrix will also be analyzed.

The results of this baseline will guide the Phase III modeling iteration where we will train a Multi-layer Perceptron Classifier (MLP) as well as a Regression Model that estimate delay duration.

## 2. Exploratory Data Analysis (EDA)
Based on our exploratory data analysis (EDA) and baseline modeling using the OTPW dataset, we observed that flight delays are influenced by temporal factors (hour of day, day of week, month), operational factors (airline, route, origin/destination airports), and weather conditions (wind speed, precipitation, visibility, pressure). We identified and removed columns with high missing values, high cardinality, or potential leakage (events occurring after departure), and imputed missing values for numerical columns with zero and categorical columns with NULL. Using the cleaned and engineered features, a baseline logistic regression model achieved 68.83% accuracy and an AUC of 68.85%, demonstrating moderate predictive power and validating the EDA insights. Moving forward, we plan to expand the EDA by joining OTPW with additional flight and weather datasets, derive new features such as graph-based route and network measures, and explore advanced models including deep learning, gradient boosting, and ensemble methods to improve predictive performance and capture complex interactions in the data.

The objective of the Phase-2 EDA is to extend the exploratory analysis conducted in Phase-1 to larger and more comprehensive datasets by incorporating the raw flight and weather data sources in addition to the OTPW dataset. This phase aims to validate earlier findings at scale, uncover new predictive signals, and prepare cleaned datasets that can support large-scale model training. Particular emphasis will be placed on ensuring data quality, understanding the joint distribution of operational and weather variables, and identifying variables that exhibit strong relationships with flight delay outcomes.

| Step | Activity                      | Description                                                                                                                                         | Deliverable                           |
| ---- | ----------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------- |
| 1    | Data Cleaning and Integration | Clean raw flight and weather datasets and perform joins using airport and timestamp information to create unified datasets.                         | Joined flight–weather dataset         |
| 2    | Dataset Construction          | Build datasets of different temporal scopes (3-month, 1-year, and multi-year) to support both exploratory analysis and model training.              | 3-month, 1-year, and 3-year datasets  |
| 3    | Predictor Identification      | Analyze correlations, delay rates, and distributions to identify **strong and moderate predictors** of flight delays in the expanded dataset.       | Candidate predictor list              |
| 4    | Scaled EDA                    | Reapply Phase-1 EDA procedures (missing data analysis, cardinality checks, correlation analysis, delay rate analysis, etc.) on the larger datasets. | Updated EDA findings                  |
| 5    | Data Normalization            | Perform cleaning, type validation, and normalization of numerical and categorical fields.                                                           | Cleaned and standardized datasets     |
| 6    | Dataset Persistence           | Store cleaned datasets for downstream modeling and experimentation.                                                                                 | Saved datasets for modeling pipelines |

The outcome of this stage will be high-quality datasets and validated exploratory insights that guide the subsequent feature engineering and modeling stages.

## 3. Feature Engineering

The goal of the feature engineering stage is to transform the cleaned datasets into model-ready feature sets that capture meaningful patterns related to flight delays. This includes handling missing values, creating derived features, and evaluating feature importance to identify variables with strong predictive signal while reducing dimensionality and avoiding overfitting.

| Step | Activity                      | Description                                                                                                                | Deliverable                  |
| ---- | ----------------------------- | -------------------------------------------------------------------------------------------------------------------------- | ---------------------------- |
| 1    | Missing Value Handling        | Impute missing values in selected columns using appropriate strategies based on variable type and distribution.            | Imputed feature dataset      |
| 2    | Feature Standardization       | Normalize or scale numeric features where required to ensure comparability across variables and improve model convergence. | Standardized features        |
| 3    | Leakage Identification        | Review and remove columns that may introduce data leakage (e.g., variables determined after departure).                    | Leakage-free dataset         |
| 4    | Derived Feature Creation      | Generate new features such as temporal transformations, categorical aggregations, graph and time-series features, or operational indicators.               | Derived feature set          |
| 5    | Interaction Features          | Create interaction and combination features that capture relationships between weather, route, and operational variables.  | Interaction features         |
| 6    | Predictive Signal Analysis    | Evaluate feature relevance using statistical measures, correlation analysis, and model-based importance metrics.           | Ranked feature list          |
| 7    | Dimensionality Reduction      | Apply techniques such as regularization or dimensionality reduction to identify a compact and informative feature subset.  | Reduced feature set          |
| 8    | Feature Dataset Checkpointing | Save the finalized feature dataset containing selected variables for model training and experimentation.                   | Checkpointed feature dataset |

This process will produce a high-quality feature set optimized for predictive modeling, which will then be used in Phase-3 to train and evaluate advanced machine learning models.

## 4. Modeling

The objective of the modeling phase is to develop predictive models that estimate the likelihood and magnitude of flight delays using the engineered features derived from the flight, weather, and airport datasets. This phase follows an iterative approach, beginning with baseline models to establish reference performance and progressively exploring more advanced algorithms capable of capturing complex relationships in the data. Model development will involve careful selection of algorithms, systematic hyperparameter tuning, and rigorous validation to ensure reliable performance and generalization.

| Step | Activity                      | Description                                                                                                                                                                                                                                                          | Deliverable                                          |
| ---- | ----------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------- |
| 1    | Baseline Classification Model | Implement a **Logistic Regression classifier** using the preprocessed datasets to predict the binary delay outcome (`DEP_DEL15`). Tune key hyperparameters such as regularization strength to optimize performance.                                                  | Baseline classification model and evaluation results |
| 2    | Regression Modeling           | Extend the modeling framework to predict the **magnitude of delays** using **Linear Regression** with the same engineered features. Evaluate performance using regression metrics and examine assumptions such as linearity and homoscedasticity.                    | Baseline regression model and performance metrics    |
| 3    | Advanced Classification Model | Develop a **Multilayer Perceptron Classifier (MLPC)** to capture non-linear relationships in the data. The model will use backpropagation for training, a logistic loss function for optimization, and the **L-BFGS optimization algorithm** to improve convergence. | Neural network classification model                  |
| 4    | Ensemble and Nonlinear Models | Explore additional algorithms such as **Random Forest** and gradient boosting methods to improve predictive performance by modeling complex feature interactions and nonlinear effects.                                                                              | Ensemble model results                               |
| 5    | Gradient Boosting Exploration | Evaluate gradient boosting approaches such as **XGBoost** for regression and classification tasks, leveraging their ability to handle heterogeneous features and capture nonlinear patterns effectively.                                                             | Boosting-based model experiments                     |
| 6    | Model Validation              | Apply **cross-validation** to evaluate model robustness and ensure reliable generalization across different data partitions.                                                                                                                                         | Cross-validation results                             |
| 7    | Overfitting Control           | Incorporate regularization techniques and monitor training vs. validation performance to mitigate overfitting and improve model stability.                                                                                                                           | Regularized models                                   |
| 8    | Feature Refinement            | Use feature selection techniques, including correlation analysis and model-based importance measures, to identify and retain the most predictive variables.                                                                                                          | Refined feature set and improved models              |

Through this iterative modeling process, we aim to identify algorithms that best capture the temporal, operational, and environmental factors influencing flight delays while ensuring that the models remain interpretable, scalable, and suitable for real-world deployment.

### 4.1 Evaluation and Reporting

The objective of the evaluation and reporting phase is to rigorously assess the performance of the predictive models developed during the project and clearly communicate the findings to stakeholders. This stage ensures that model results are interpreted in the context of the business problem, predicting flight departure delays and that insights derived from the analysis are presented in a structured and reproducible manner. In addition to measuring model performance using standard machine learning metrics, this phase focuses on comparing alternative models, validating results across different datasets, and documenting the methodology, results, and key insights for the final project deliverables.

| Step | Activity                     | Description                                                                                                                                                                  | Deliverable                           |
| ---- | ---------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------- |
| 1    | Define Evaluation Metrics    | Select appropriate metrics aligned with the problem formulation, such as **Accuracy, AUC-ROC, Precision, Recall, and F1-score**, to measure classification performance.      | Evaluation framework                  |
| 2    | Model Performance Evaluation | Evaluate trained models on the held-out test dataset using the selected metrics to assess predictive accuracy and generalization.                                            | Model performance results             |
| 3    | Model Comparison             | Compare baseline and advanced models (e.g., logistic regression, ensemble methods, deep learning models) to determine the best-performing approach.                          | Comparative model analysis            |
| 4    | Error Analysis               | Analyze misclassified predictions to understand model limitations and identify patterns such as specific routes, weather conditions, or time periods associated with errors. | Error analysis insights               |
| 5    | Visualization of Results     | Generate visualizations such as **ROC curves, confusion matrices, feature importance plots, and model performance summaries** to communicate results clearly.                | Model evaluation visualizations       |
| 6    | Interpretation of Findings   | Interpret model outputs and feature importance to understand the key drivers of flight delays and validate insights discovered during EDA.                                   | Key insights and conclusions          |
| 7    | Documentation and Reporting  | Compile all results, methodologies, and insights into structured documentation and reports for project review and presentation.                                              | Final report and supporting materials |
| 8    | Presentation Preparation     | Prepare visual summaries, diagrams, and key takeaways to effectively communicate project outcomes in the final presentation.                                                 | Project presentation                  |

The outcome of this phase will be a comprehensive evaluation of model performance, along with clear documentation and visualizations that summarize the analytical process, highlight the most influential predictors of flight delays, and provide actionable insights for improving delay prediction systems.

# Appendix / Reference notebooks

Reference notebook:
1. [OTPW EDA](https://dbc-fae72cab-cf59.cloud.databricks.com/editor/notebooks/2181853686874581?o=4021782157704243)
2. [OTPW EDA - Data Analysis](https://dbc-fae72cab-cf59.cloud.databricks.com/editor/notebooks/3130150146287200?o=4021782157704243)
3. [OTPW EDA - Candidate Feature Analysis](https://dbc-fae72cab-cf59.cloud.databricks.com/editor/notebooks/2181853686874027?o=4021782157704243)
4. [Flights EDA](https://dbc-fae72cab-cf59.cloud.databricks.com/editor/notebooks/2181853686874569?o=4021782157704243)
5. [Weather EDA](https://dbc-fae72cab-cf59.cloud.databricks.com/editor/notebooks/3684661422687193?o=4021782157704243)